# Lab primer: Print, tables, and advanced queries

**Data 6** — Labs **3** (print, arrays, `Table` basics, line plot), **8** (`where` / `group` / `pivot` / `join` on movies), and a bit of **5** (`bar` / `barh`). Work top to bottom. CSVs must sit next to this notebook: `unemployment.csv`, `movies.csv`.


In [ ]:
# Run this cell once — it loads the libraries we use in Data 6
import numpy as np
from datascience import *

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")

import warnings
warnings.simplefilter("ignore")


---

# Part 1 — Print and arrays

`print` writes to the output area (good for debugging). The last expression in a cell can also show as `Out[ ]`; `print` returns `None`. Run the next cell, then do the exercise.


In [ ]:
print("Hello, World!")
print("First line")
print("Second line")


---

## Exercise 1.1 — first and last in an array

Given `arr`, print the **first** and **last** value on separate lines. Use indices on `arr` (last via **negative** indexing). Expected for the sample: `1` then `7`.


In [ ]:
arr = make_array(1, 3, 5, 7)

# Replace FIRST_INDEX and LAST_INDEX with integers (use negative indexing for the last)
FIRST_INDEX = ...
LAST_INDEX = ...
print(arr.item(FIRST_INDEX))
print(arr.item(LAST_INDEX))


---
# Part 2 — Load and inspect a table

A **`Table`** has rows (records) and columns (variables). Run the next cells: **load** `unemployment.csv`, **`show(5)`** (always pass a small integer), then **`num_rows`** and **`num_columns`**.


In [ ]:
unemployment_rates = Table.read_table("unemployment.csv")

In [ ]:
unemployment_rates.show(5)

In [ ]:
unemployment_rates.num_rows, unemployment_rates.num_columns

---

## Exercise 2.1 — `select`

**`select`** — copy of the table with only the listed columns (in order). Build `unemployment_totals` with `"Month"`, `"Year"`, `"Total"`.


In [ ]:
unemployment_totals = unemployment_rates.select("...", "...", "...")
unemployment_totals


---

## Exercise 2.2 — `sort`

**`sort`** — sorted copy; use `descending=True` for largest first. Sort by `"Total"` so the highest rate is first.


In [ ]:
unemployment_rate_highest = unemployment_rates.sort("...", descending=True)
unemployment_rate_highest.show(5)


---

## Exercise 2.3 — `where`

**`where`** — copy with rows that match. Shorthand: `tbl.where("Year", 2025)` means that column equals `2025`. Build `unemployment_2025`.


In [ ]:
unemployment_2025 = unemployment_rates.where("...", 2025)
unemployment_2025


---

# Part 3 — Time on the x-axis

Plotting `"Total"` vs `"Month"` mixes different years on the same x positions. Build **`times`** = year + month fraction: `years + (months - 1) / 12`, then **`with_columns`** + **`plot`**.


In [ ]:
months = unemployment_rates.column("...")
years = unemployment_rates.column("...")

# Pretty-print check (run after you fill in the column names above)
Table().with_columns("Month", months, "Year", years).show(5)


Fill in `times` so it equals `years + (months - 1) / 12` (January → `.0` of that year).


In [ ]:
# Replace ... so this matches: years + (months - 1) / 12
times = years + (months - ...) / ...
Table().with_columns("Month", months, "Year", years, "Time", times).show(5)


Build `unemployment_over_time` with `"Time"` (your `times`) and `"Total Unemployment"` (`unemployment_rates.column("Total")`), then plot.


In [ ]:
unemployment_over_time = Table().with_columns(
    "Time", ...,
    "Total Unemployment", unemployment_rates.column("..."),
)
unemployment_over_time.plot("Time", "Total Unemployment")


---

# Part 4 — `where` + `are` on movies

Load and inspect `movies` below.


In [ ]:
movies = Table.read_table("movies.csv")
movies.show(5)


<hr style="border: 1px solid #fdb515;" />

## The [`where`](https://www.data8.org/datascience/_autosummary/datascience.tables.Table.where.html#datascience.tables.Table.where) method

Keeps rows where a column passes a test. Second argument: an **`are`** predicate, or a single value for **exact match** (`where("col", x)` ≡ `where("col", are.equal_to(x))`). More predicates: [datascience predicates](http://data8.org/datascience/predicates.html). Prefix with **`not_`** for the opposite test.

| Method | Type | Meaning |
| --- | --- | --- |
| `are.equal_to(n)` | number | equals `n` |
| `are.above(n)` / `are.below(n)` | number | above / below `n` |
| `are.above_or_equal_to(n)` / `are.below_or_equal_to(n)` | number | inclusive bounds |
| `are.containing(s)` | string | substring in cell |
| `are.containined_in(s)` | string | cell’s string contained in `s` |

Example — gross over 200 (run next cell):


In [ ]:
movies.where("Worldwide Gross (Millions)", are.above(200))

---
## Exercise 4.1 — years 2006–2009

Chain two **`where`** calls on `"Year"` with `are.above_or_equal_to(2006)` and `are.below_or_equal_to(2009)`. The next cell is a completed example—try writing it yourself first if you want practice.


In [ ]:
late_2000s = (
    movies.where("Year", are.above_or_equal_to(...))
    .where("Year", ...(2009))
)
late_2000s.show(5)

---

# Part 5 — `group`

<hr style="border: 1px solid #fdb515;" />

## The [`group`](http://data8.org/datascience/_autosummary/datascience.tables.Table.group.html#datascience.tables.Table.group) method

Puts rows into **bins** by one column’s value. **`group(col)`** counts rows per bin; **`group(col, fn)`** (e.g. `np.mean`) aggregates numeric columns per bin (suffix like `mean` on new names). Original table unchanged unless reassigned. [Table visualizer — group](http://www.data8.org/interactive_table_functions/).

Example: counts by `"Genre"` (run next cell).


In [ ]:
movies_by_genre = movies.group("Genre")
movies_by_genre


### `bar` and `barh` on a small table

**`barh`** — horizontal bars for **categorical** labels (*ordinal* vs *nominal*: [notes](https://data6.org/notes/05-variables/#variable-types)). One required argument: the **category column**. **`bar`** — same idea, **vertical** bars. [barh docs](http://data8.org/datascience/_autosummary/datascience.tables.Table.barh.html#datascience.tables.Table.barh) · [bar docs](https://www.data8.org/datascience/_autosummary/datascience.tables.Table.bar.html#datascience.tables.Table.bar)

Use a **small** table (e.g. after **`group`**), not every raw row. Run `movies_by_genre.bar("Genre")` next.


In [ ]:
# Vertical bar chart: one bar per genre, bar height = number of movies in that genre
movies_by_genre.bar("Genre")


---

## Exercise 5.1 — `group` by quality

`quality_groups = movies.group("...")` with the quality column.


In [ ]:
quality_groups = movies.group("...")
quality_groups


**Second argument to `group`** — e.g. `np.mean` averages each numeric column per bin; use **`select`** to keep the columns you want. Run the next cell.


In [ ]:
quality_group_ratings = movies.group("Quality", np.mean)
quality_ratings = quality_group_ratings.select("Quality", "Audience score % mean")
quality_ratings


---

## Exercise 5.2 — mean gross by quality

Two columns only: `"Quality"` and `"Worldwide Gross (Millions) mean"` (`group` with `np.mean`, then `select`).


In [ ]:
money_by_quality = movies.group("...", np.mean).select("...", "...")
money_by_quality


---

# Part 6 — `pivot`

<hr style="border: 1px solid #fdb515;" />

## The [`pivot`](http://data8.org/datascience/_autosummary/datascience.tables.Table.pivot.html#datascience.tables.Table.pivot) method

The `pivot` method allows us to see the *intersection* of two of our column labels. `pivot` essentially sorts the contents of the dataset based on the **combination** of the two column labels you pivot on. All the table's rows that share values in those two columns go into the same bin, and that happens for **every** combination of the first and second pivot columns.

The `pivot` method has **4** important arguments, **2** mandatory and **2** optional:

| **Argument** | **Description** |
| :--- | :--- |
| `columns` | The label whose unique values will appear as the **columns** of the output pivot table |
| `rows` | The label whose unique values will appear as the **rows** of the output pivot table |
| *Optional:* `values` | Values to use when aggregating |
| *Optional:* `collect` | Function used to aggregate the `values` provided in the previous argument |

You must use `values` and `collect` **together**—one does not work without the other.

> For a visualization of `.pivot`, see the [Data 8 Table Visualizer](http://www.data8.org/interactive_table_functions/).

In code, the first two arguments are `pivot(columns, rows, ...)` — **columns first, then rows.** With no `values`/`collect`, each cell is a **count** of rows in that row–column pair. **Read a cell** as: how many rows (or the aggregated value, if you passed `values` and `collect`) have *this* row label and *this* column label? Example: `movies.pivot("Genre", "Quality")` (next cell).


In [ ]:
genre_quality_pivot = movies.pivot("Genre", "Quality")
genre_quality_pivot


---

## Exercise 6.1 — pivot total gross

Rows = studios, columns = genres, cells = **sum** of `"Worldwide Gross (Millions)"`. Fill the three `"..."` labels; `sum` is already set.


In [ ]:
column_label = "..."   # pivot columns → genres
row_label = "..."      # pivot rows → studios
value_to_collect = "..."
collection_function = sum   # keep as sum (adds gross in each cell)

studio_genre_total_gross = movies.pivot(column_label, row_label, value_to_collect, collection_function)
studio_genre_total_gross


---

# Part 7 — `join`

<hr style="border: 1px solid #fdb515;" />

## The [`join`](https://www.data8.org/datascience/_autosummary/datascience.tables.Table.join.html#datascience.tables.Table.join) method

Attach rows from **`other`** when keys match: **`left.join("KeyCol", other)`** if names align; else **`left.join("LeftCol", other, "RightCol")`**. 

| Arg | Role |
| --- | --- |
| `column_label` | join key on the **left** table |
| `other` | right `Table` |
| `other_label` (opt.) | key column **in `other`** if the name differs |

Example: `dogs` and `owners` below.


In [ ]:
dogs = Table().with_columns(
    "Name", np.array(["Spot", "Rex", "Fluffy", "Doge"]),
    "Breed", np.array(["Golden Retriever", "Cockapoo", "Corgi", "Coin"]),
    "Owner", np.array(["James", "Will", "Josh", "Sandra"]),
)
dogs


In [ ]:
owners = Table().with_columns(
    "Owner", np.array(["James", "Josh", "Sandra", "Will"]),
    "Owner Age", np.array([18, 21, 20, 21]),
)
owners

Shared key `"Owner"`: **`dogs.join("Owner", owners)`** (next cell).


In [ ]:
doggy_data = dogs.join("Owner", owners)
doggy_data


Different key name on the right: pass **`other_label`**, e.g. **`dogs.join("Owner", owners_new_label, "Name")`** after `relabeled`.


In [ ]:
owners_new_label = owners.relabeled("Owner", "Name")
doggy_data_fixed = dogs.join("Owner", owners_new_label, "Name")
doggy_data_fixed


---

## Exercise 7.1 — `join`

`my_doggy_data` = same result as `doggy_data` (fill the join key string).


In [ ]:
my_doggy_data = dogs.join("...", owners)
my_doggy_data


---
## Summary — table methods (quick reference)

| **Name** | **Example** | **Purpose** |
| :--- | :--- | :--- |
| `sort` | `tbl.sort("N")` | Copy sorted by one column (`descending=True` for reverse). |
| `where` | `tbl.where("N", are.above(2))` | Copy keeping rows that match a *predicate* or exact value. |
| `num_rows` / `num_columns` | `tbl.num_rows` | Size of the table. |
| `select` / `drop` | `tbl.select("N")` / `tbl.drop("N")` | Keep only listed columns / drop listed columns (new table). |
| `read_table` | `Table.read_table("f.csv")` | Build a table from a CSV file. |
| `show` | `tbl.show(5)` | Show the first *n* rows (always pass *n*). |
| `column` | `tbl.column("N")` | One column as an array. |
| `with_columns` | `tbl.with_columns("L", arr)` | Add or replace columns (new table). |
| `group` | `tbl.group("N")`, `tbl.group("N", np.mean)` | Count rows per category, or aggregate with a function. |
| `pivot` | `tbl.pivot(cols, rows, val, fn)` | Two-way grid of counts or aggregated values. |
| `join` | `left.join("K", right, ...)` | Merge two tables on matching key columns. |
| `relabeled` | `tbl.relabeled("A","B")` | Copy with one column renamed. |
| `bar` / `barh` | `tbl.bar("Cat")` | Bar chart for categories (use a small aggregated table). |
| `plot` | `tbl.plot("x","y")` | Line plot (`%matplotlib inline` required). |


---

## Done

Restart kernel and **Run All** if outputs look wrong. Sources: Labs 3, 5, 8.
